In [1]:
import importlib
import io
from contextlib import redirect_stdout

import pandas as pd
import matplotlib.pyplot as plt

import Modelos
import Estrategia as Estrategias

from UniversoActivos import UniversoActivosDinamico
from ProveedorDatos import YFinanceProvider
from VariablesTransformation import FeatureEngineer
from Backtest import BacktestEngine

RandomForestModel = Modelos.RandomForestModel
XGBoostModel = Modelos.XGBoostModel

EstrategiaMLEquiponderada = Estrategias.EstrategiaMLEquiponderada
EstrategiaMLMinVarAlphaTilt = Estrategias.EstrategiaMLMinVarAlphaTilt
import itertools


In [2]:
import pandas as pd
import requests

def get_eurostoxx50_tickers():
    url = 'https://en.wikipedia.org/wiki/EURO_STOXX_50'
    
    # Añadimos una cabecera para simular un navegador real
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36"
    }
    
    # Hacemos la petición con requests
    response = requests.get(url, headers=headers)
    
    # Pasamos el contenido HTML (response.text) a pandas
    tables = pd.read_html(response.text)
    
    # Buscamos la tabla que contiene la columna 'Ticker'
    df = next(table for table in tables if 'Ticker' in table.columns)
    
    return df['Ticker'].tolist()

tickers = get_eurostoxx50_tickers()
print(f"Total empresas: {len(tickers)}")
print(f"Muestra: {tickers}")

Total empresas: 50
Muestra: ['ADS.DE', 'ADYEN.AS', 'AD.AS', 'AI.PA', 'AIR.PA', 'ALV.DE', 'ABI.BR', 'ARGX.BR', 'ASML.AS', 'CS.PA', 'BAS.DE', 'BAYN.DE', 'BBVA.MC', 'SAN.MC', 'BMW.DE', 'BNP.PA', 'BN.PA', 'DBK.DE', 'DB1.DE', 'DHL.DE', 'DTE.DE', 'ENEL.MI', 'ENI.MI', 'EL.PA', 'RACE.MI', 'RMS.PA', 'IBE.MC', 'ITX.MC', 'IFX.DE', 'INGA.AS', 'ISP.MI', 'OR.PA', 'MC.PA', 'MBG.DE', 'MUV2.DE', 'NDA-FI.HE', 'PRX.AS', 'RHM.DE', 'SAF.PA', 'SGO.PA', 'SAN.PA', 'SAP.DE', 'SU.PA', 'SIE.DE', 'ENR.DE', 'TTE.PA', 'DG.PA', 'UCG.MI', 'VOW.DE', 'WKL.AS']


C:\Users\jpuerta\AppData\Local\Temp\ipykernel_127132\1949759624.py:16: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  tables = pd.read_html(response.text)


In [4]:
start_date = "2025-09-03"
end_date = "2026-03-11"
universo = UniversoActivosDinamico(tickers_actuales=tickers, start_date=start_date, end_date=end_date, csv_cambios_path=r"eurostoxx50_historico_cambios.csv")
proveedor = YFinanceProvider()
fe = FeatureEngineer(criterio=5, ticker_indice="SPY")


In [ ]:
# Define ranges de hiperparámetros relevantes
param_grid = {
    "n_estimators": [100, 200],
    "max_depth": [3, 4],
    "lambda_risk": [2.0, 4.0],
    "w_max": [0.2],
    "turnover_max": [0.15],
    "p_neutral": [0.55],
    "alpha_scale": [1.0]
}

# Genera todas las combinaciones
combinations = list(itertools.product(*param_grid.values()))

results = []

for combo in combinations:
    params = dict(zip(param_grid.keys(), combo))
    modelo = RandomForestModel(
        n_estimators=params["n_estimators"],
        max_depth=params["max_depth"],
        class_weight=None,
        random_state=42,
        positive_class_weight=5.0
    )
    estrategia = EstrategiaMLMinVarAlphaTilt(
        modelo=modelo,
        n_activos_obj=10,
        umbral_salida=15,
        p_neutral=params["p_neutral"],
        alpha_scale=params["alpha_scale"],
        lambda_risk=params["lambda_risk"],
        lambda_tc=0.001,
        w_max=params["w_max"],
        turnover_max=params["turnover_max"],
        no_trade_band=0.002,
        coste_transaccion=0.0005,
        utility_buffer=0.0001,
        min_hist_obs=26
    )
    engine = BacktestEngine(
        universo=universo,
        proveedor=proveedor,
        feature_engineer=fe,
        estrategia=estrategia,
        start_date=start_date,
        end_date=end_date,
        len_ventana=4,
        nominal=10000000
    )
    fechas, serie_estrategia, metrics_view = engine.print_results(bmks=["^STOXX50E"], plot=False)
    # Extrae la métrica que quieras optimizar, por ejemplo "Rentabilidad anualizada"
    rentabilidad = float(metrics_view.loc["Rentabilidad anualizada", "Estrategia"].strip('%')) / 100
    results.append((params, rentabilidad, metrics_view))

# Ordena los resultados por rentabilidad anualizada (o la métrica que prefieras)
results_sorted = sorted(results, key=lambda x: x[1], reverse=True)

# Muestra los mejores parámetros y sus métricas
for i, (params, rentabilidad, metrics_view) in enumerate(results_sorted[:5]):  # Top 5
    print(f"Rank {i+1}:")
    print("Params:", params)
    print("Rentabilidad anualizada:", rentabilidad)
    display(metrics_view)
    print("-" * 40)

In [5]:
param_grid = {
    "n_estimators": [100, 180, 300],
    "max_depth": [3, 4],
    "n_activos_obj": [5, 10],
    "umbral_salida": [10, 15]
}

combinations = list(itertools.product(*param_grid.values()))
results2 = []

for combo in combinations:
    params = dict(zip(param_grid.keys(), combo))
    modelo = RandomForestModel(
        n_estimators=params["n_estimators"],
        max_depth=params["max_depth"],
        class_weight=None,
        random_state=42,
        positive_class_weight=5.0
    )
    estrategia = EstrategiaMLEquiponderada(
        modelo=modelo,
        n_activos_obj=params["n_activos_obj"],
        umbral_salida=params["umbral_salida"]
    )
    engine = BacktestEngine(
        universo=universo,
        proveedor=proveedor,
        feature_engineer=fe,
        estrategia=estrategia,
        start_date=start_date,
        end_date=end_date,
        len_ventana=4,
        nominal=10000000
    )
    fechas, serie_estrategia, metrics_view = engine.print_results(bmks=["^STOXX50E"], plot=False)
    # Extrae la métrica que quieras optimizar, por ejemplo "Rentabilidad anualizada"
    rentabilidad = float(metrics_view.loc["Rentabilidad anualizada", "Estrategia"].strip('%')) / 100
    results2.append((params, rentabilidad, metrics_view))

# Ordena los resultados por rentabilidad anualizada (o la métrica que prefieras)
results2_sorted = sorted(results2, key=lambda x: x[1], reverse=True)

# Muestra los mejores parámetros y sus métricas
for i, (params, rentabilidad, metrics_view) in enumerate(results2_sorted[:5]):  # Top 5
    print(f"Rank {i+1}:")
    print("Params:", params)
    print("Rentabilidad anualizada:", rentabilidad)
    display(metrics_view)
    print("-" * 40)

[*********************100%***********************]  53 of 53 completed
[*********************100%***********************]  53 of 53 completed
c:\Users\jpuerta\OneDrive - Analistas Financieros Internacionales (Afi)\Documentos\Master Quant\Gestión de activos\Trabajo gestión cuantitativa\Trabajo_gestion_cuantitativa\ProveedorDatos.py:72: FutureWarning: DataFrameGroupBy.resample operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  weekly = weekly.agg(


Modelo entrenado en fecha 2025-09-03


[*********************100%***********************]  1 of 1 completed

Modelo entrenado en fecha 2026-03-04



[*********************100%***********************]  53 of 53 completed
[*********************100%***********************]  53 of 53 completed
c:\Users\jpuerta\OneDrive - Analistas Financieros Internacionales (Afi)\Documentos\Master Quant\Gestión de activos\Trabajo gestión cuantitativa\Trabajo_gestion_cuantitativa\ProveedorDatos.py:72: FutureWarning: DataFrameGroupBy.resample operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  weekly = weekly.agg(


...

----------------------------------------
Rank 4:
Params: {'n_estimators': 180, 'max_depth': 4, 'n_activos_obj': 10, 'umbral_salida': 10}
Rentabilidad anualizada: 0.1782


,Estrategia,^STOXX50E
Rentabilidad total,13.08%,9.62%
Rentabilidad anualizada,17.82%,13.03%
Volatilidad anualizada,13.35%,11.60%
Sharpe,1.30,1.11
Sortino,1.87,1.63
Max Drawdown,-7.01%,-7.91%
Calmar,2.54,1.65
Win rate,41.27%,35.98%
Mejor periodo,2.41%,2.67%
Peor periodo,-2.87%,-3.59%


----------------------------------------
Rank 5:
Params: {'n_estimators': 100, 'max_depth': 3, 'n_activos_obj': 10, 'umbral_salida': 15}
Rentabilidad anualizada: 0.1778


,Estrategia,^STOXX50E
Rentabilidad total,13.06%,9.62%
Rentabilidad anualizada,17.78%,13.03%
Volatilidad anualizada,13.56%,11.60%
Sharpe,1.27,1.11
Sortino,1.84,1.63
Max Drawdown,-7.38%,-7.91%
Calmar,2.41,1.65
Win rate,40.74%,35.98%
Mejor periodo,2.49%,2.67%
Peor periodo,-3.10%,-3.59%


----------------------------------------


In [ ]:
len(results2)

81

In [ ]:
import pickle

# Guardar
with open("results2.pkl", "wb") as f:
    pickle.dump(results2, f)

# Cargar
with open("results2.pkl", "rb") as f:
    results2_recuperado = pickle.load(f)